# Base network from ChEA databases (BooleaBayes)

Build a base transcription-factor network from curated ChEA (and related) target interactions — the original BooleaBayes approach — for systems without scATAC-seq data.

> **Starter notebook.** The cells below are a scaffold: real function calls with placeholder paths/arguments and `TODO` markers. Fill in your own data and run top-to-bottom. Anything marked `TODO` is a choice you need to make for your dataset.

## Setup and imports

In [ ]:
import os
import os.path as op
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import bobaT as bb

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
import warnings
warnings.filterwarnings('ignore')

## Configure paths and transcription factors

In [ ]:
# Input data
DATA_DIR = './test_data'

# Output directories
OUTPUT_DIR = './output'
VAL_DIR = './output/validation'
ATTRACTOR_DIR = './output/attractors'
PERTURB_DIR = './output/perturbations'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# TODO: the transcription factors of interest for your system
tfs = ['Ascl1', 'Neurod1', 'Pou2f3', 'Yap1']  # example

# Enrichr libraries use human gene symbols, so query in upper case
# (for mouse data, map back to your data's gene names after building the network)
tfs = [tf.upper() for tf in tfs]

## Build the network with `make_network`

`bb.net.make_network` queries Enrichr for each TF in `tfs` and adds an edge `source -> TF` whenever another TF in your list is reported as a regulator of it in the ChEA (2013/2015/2016), ENCODE TF ChIP-seq (2014/2015), TRANSFAC/JASPAR PWM, or ENCODE+ChEA consensus libraries. Each edge records which databases support it.

By default (`libraries='current'`), it searches the current Enrichr TF-target libraries: ChEA 2013–2022, ENCODE TF ChIP-seq 2014/2015, ENCODE+ChEA consensus, TRANSFAC/JASPAR PWMs, JASPAR 2025 (human and mouse) and TRRUST 2019. `libraries='legacy'` reproduces networks built before November 2026, which used the original library list and parsing. With today's Enrichr term names, only ChEA_2013 and TRANSFAC_and_JASPAR_PWMs yield edges in that mode. You can also pass a list of library names; see `bb.enrichr.LIBRARY_PARSERS` for the supported ones.

The ChEA versions overlap heavily, so one ChIP-seq experiment can appear in all four and pass the "two or more databases" filter alone. `collapse_db_versions=True` counts all ChEA versions (and the ENCODE+ChEA consensus), both ENCODE versions, and all PWM libraries each as one source. An edge then needs two independent kinds of evidence. This is stricter, and with a small TF list it can prune the network down to nothing.

It then writes one CSV per pruning level to `outdir`:

| File | Contents |
|---|---|
| `<network_name>_unfiltered.csv` | All edges found (only if `save_unfiltered=True`) |
| `<network_name>_pruned.csv` | Source and/or sink nodes removed iteratively (`do_prune`, `prune_sources`, `prune_sinks`) |
| `<network_name>_high_evidence.csv` | Edges supported by at least two databases (`do_prune_info`) |
| `<network_name>_chea.csv` | Edges supported by a ChEA database (`do_prune_to_chea`) |

The returned graph is the most heavily pruned network. Queries are spaced by one second per TF, so large TF lists take a few minutes.

In [ ]:
# TODO: choose which pruning steps to apply for your system
G = bb.net.make_network(
    tfs,
    outdir=OUTPUT_DIR,
    network_name='chea_network',
    libraries='current',    # default; 'legacy' = original libraries and parsing
    save_unfiltered=True,   # also keep the unpruned network
    do_prune=True,          # remove source/sink nodes
    prune_sources=True,
    prune_sinks=True,
    do_prune_info=True,     # keep edges with evidence in >= 2 databases
    collapse_db_versions=False,  # True: >= 2 independent sources (ChEA versions count once)
    do_prune_to_chea=True,  # keep edges with ChEA evidence
    prune_self_loops=True,
)
print(f'{G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## Inspect the network levels

Compare how many edges survive each pruning step before choosing which file to use downstream. Pruning too aggressively can remove TFs you care about.

In [ ]:
for level in ['unfiltered', 'pruned', 'high_evidence', 'chea']:
    path = op.join(OUTPUT_DIR, f'chea_network_{level}.csv')
    if op.exists(path) and os.path.getsize(path) > 0:
        edges = pd.read_csv(path, header=None, names=['source', 'target'])
        n_nodes = len(set(edges['source']) | set(edges['target']))
        print(f'{level:>14}: {n_nodes} nodes, {len(edges)} edges')
    else:
        print(f'{level:>14}: (not written or empty)')

# Supporting databases for each edge of the returned network
pd.DataFrame([(u, v, ', '.join(d.get('db', []))) for u, v, d in G.edges(data=True)],
             columns=['source', 'target', 'databases'])

## Load the network for rule inference

Load the CSV you chose with `bb.load.load_network`, which returns the graph and vertex dictionary used in the inference tutorials.

In [ ]:
# TODO: pick the pruning level to use
network_file = op.join(OUTPUT_DIR, 'chea_network_chea.csv')
graph, vertex_dict = bb.load.load_network(network_file, remove_sinks=False,
                                          remove_selfloops=False, remove_sources=False)
v_names, nodes = bb.utils.get_nodes(vertex_dict, graph)
print(len(nodes), 'nodes:', sorted(nodes))

## Next step

Use this network for [rule inference](inference_two_timepoints.ipynb).